## Weather Data Analysis for Nebraska Fields

This notebook analyzes historical weather data from NASA POWER (Prediction of Worldwide Energy Resources) for 5 agricultural fields in Nebraska.

### Data Source
- **Provider**: NASA POWER REST API
- **Endpoint**: https://power.larc.nasa.gov/api/temporal/daily/point
- **Date Range**: January 1 - December 31, 2025
- **Resolution**: 0.5° grid (~55km)

### Field Coordinates

| Field ID | Latitude | Longitude | Location |
|----------|----------|-----------|----------|
| field_001 | 40.92336 | -96.3735 | Near Crete, NE (Saline County) |
| field_002 | 40.92459 | -96.38387 | Near Crete, NE (Saline County) |
| field_003 | 40.91862 | -96.36465 | Near Crete, NE (Saline County) |
| field_004 | 40.95425 | -96.0716 | Near Ashland, NE (Saunders County) |
| field_005 | 41.01814 | -96.07927 | Near Ashland, NE (Saunders County) |

### Parameters Retrieved

| Parameter | Description | Units |
|-----------|-------------|-------|
| T2M | Daily mean temperature at 2m | °C |
| T2M_MAX | Daily maximum temperature at 2m | °C |
| T2M_MIN | Daily minimum temperature at 2m | °C |
| PRECTOTCORR | Precipitation (bias-corrected) | mm/day |
| ALLSKY_SFC_SW_DWN | Solar radiation | MJ/m²/day |
| RH2M | Relative humidity at 2m | % |
| WS10M | Wind speed at 10m | m/s |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from pathlib import Path

DATA_DIR = Path('data')
weather_file = DATA_DIR / 'weather_5fields_2025.csv'
stats_file = DATA_DIR / 'weather_stats.csv'

## Load Weather Data

In [ ]:
# Load daily weather data
weather = pd.read_csv(weather_file, parse_dates=['date'])
print(f"Loaded {len(weather)} records")
print(f"Date range: {weather['date'].min().date()} to {weather['date'].max().date()}")
print(f"Fields: {weather['field_id'].unique().tolist()}")
print(f"\nColumns: {weather.columns.tolist()}")

In [ ]:
# Display sample data
weather.head(10)

## Data Summary

In [ ]:
# Basic statistics
weather.describe()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(weather.isnull().sum())

## Temperature Analysis

Plot daily temperature (mean, min, max) for each field throughout 2025.

In [ ]:
# Plot temperature for each field
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

fields = sorted(weather['field_id'].unique())

for idx, field_id in enumerate(fields):
    ax = axes[idx]
    field_data = weather[weather['field_id'] == field_id].sort_values('date')
    
    # Fill between min and max
    ax.fill_between(field_data['date'], field_data['T2M_MIN'], field_data['T2M_MAX'],
                   alpha=0.3, label='Min/Max range', color='steelblue')
    # Plot mean
    ax.plot(field_data['date'], field_data['T2M'], linewidth=0.8, label='Mean', color='navy')
    # Reference lines
    ax.axhline(0, color='grey', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.axhline(10, color='green', linewidth=0.5, linestyle='--', alpha=0.5, label='GDD Base (10°C)')
    
    ax.set_ylabel('Temperature (°C)')
    ax.set_title(f'Daily Temperature — {field_id}')
    ax.legend(loc='upper right', fontsize=8)
    ax.xaxis.set_major_formatter(DateFormatter('%b'))
    ax.grid(True, alpha=0.3)

# Hide the empty subplot
axes[-1].axis('off')

plt.tight_layout()
plt.savefig(DATA_DIR / 'temperature_all_fields.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {DATA_DIR / 'temperature_all_fields.png'}")

## Growing Degree Days (GDD) Calculation

GDD measures heat accumulation for crop development.

**Formula:** `GDD = max(0, min(T_avg, cap) - base)`

Where:
- T_avg = (T2M_MAX + T2M_MIN) / 2
- Base temperature = 10°C (corn)
- Cap temperature = 30°C

In [ ]:
# Calculate GDD
def calculate_gdd(df, base_temp=10.0, cap_temp=30.0):
    """Calculate daily and cumulative GDD."""
    out = df.copy()
    t_avg = ((out['T2M_MIN'] + out['T2M_MAX']) / 2).clip(upper=cap_temp)
    out['gdd'] = (t_avg - base_temp).clip(lower=0)
    out['gdd_cumulative'] = out.groupby('field_id')['gdd'].cumsum()
    return out

weather_gdd = calculate_gdd(weather, base_temp=10.0, cap_temp=30.0)

# Show sample with GDD
weather_gdd[['field_id', 'date', 'T2M_MIN', 'T2M_MAX', 'T2M', 'gdd', 'gdd_cumulative']].head(10)

## GDD Accumulation by Field

In [ ]:
# Plot cumulative GDD
fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.tab10(np.linspace(0, 1, len(fields)))

for idx, field_id in enumerate(sorted(fields)):
    field_data = weather_gdd[weather_gdd['field_id'] == field_id].sort_values('date')
    ax.plot(field_data['date'], field_data['gdd_cumulative'], 
           label=field_id, color=colors[idx], linewidth=1.5)

ax.set_ylabel('Cumulative GDD')
ax.set_xlabel('Date')
ax.set_title('Growing Degree Day Accumulation (2025) — Base 10°C, Cap 30°C')
ax.legend()
ax.xaxis.set_major_formatter(DateFormatter('%b'))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / 'gdd_accumulation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {DATA_DIR / 'gdd_accumulation.png'}")

In [ ]:
# Total GDD by field
final_gdd = weather_gdd.groupby('field_id')['gdd'].sum().reset_index()
final_gdd.columns = ['field_id', 'total_gdd']
print("Total GDD by field (2025):")
print(final_gdd.to_string(index=False))

## Precipitation Analysis

In [ ]:
# Calculate cumulative precipitation
weather_precip = weather.copy()
weather_precip['precip_cumulative'] = weather_precip.groupby('field_id')['PRECTOTCORR'].cumsum()

# Plot cumulative precipitation
fig, ax = plt.subplots(figsize=(12, 6))

for idx, field_id in enumerate(sorted(fields)):
    field_data = weather_precip[weather_precip['field_id'] == field_id].sort_values('date')
    ax.plot(field_data['date'], field_data['precip_cumulative'], 
           label=field_id, color=colors[idx], linewidth=1.5)

ax.set_ylabel('Cumulative Precipitation (mm)')
ax.set_xlabel('Date')
ax.set_title('Cumulative Precipitation (2025)')
ax.legend()
ax.xaxis.set_major_formatter(DateFormatter('%b'))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / 'precipitation_cumulative.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {DATA_DIR / 'precipitation_cumulative.png'}")

## Monthly Precipitation Distribution

In [ ]:
# Calculate monthly precipitation
weather_monthly = weather.copy()
weather_monthly['month'] = weather_monthly['date'].dt.month
weather_monthly['year'] = weather_monthly['date'].dt.year

monthly_precip = weather_monthly.groupby(['field_id', 'month'])['PRECTOTCORR'].sum().reset_index()

# Pivot for plotting
monthly_pivot = monthly_precip.pivot(index='month', columns='field_id', values='PRECTOTCORR').fillna(0)

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Plot monthly precipitation
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(12)
width = 0.15

for idx, field_id in enumerate(sorted(fields)):
    precip = monthly_pivot[field_id].values if field_id in monthly_pivot.columns else np.zeros(12)
    ax.bar(x + idx * width, precip, width, label=field_id, color=colors[idx])

ax.set_ylabel('Precipitation (mm)')
ax.set_xlabel('Month')
ax.set_title('Monthly Precipitation (2025)')
ax.set_xticks(x + width * (len(fields) - 1) / 2)
ax.set_xticklabels(months)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(DATA_DIR / 'precipitation_monthly.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {DATA_DIR / 'precipitation_monthly.png'}")

## Seasonal Summary

In [ ]:
# Calculate seasonal summary (growing season: April-September)
growing_season = weather_gdd[weather_gdd['month'].isin([4, 5, 6, 7, 8, 9])]

summary = growing_season.groupby('field_id').agg({
    'T2M': ['mean', 'min', 'max'],
    'PRECTOTCORR': 'sum',
    'ALLSKY_SFC_SW_DWN': 'sum',
    'RH2M': 'mean',
    'gdd': 'sum'
}).round(2)

summary.columns = ['avg_temp_C', 'min_temp_C', 'max_temp_C', 'precip_mm', 'solar_mj', 'humidity_pct', 'gdd_total']
summary = summary.reset_index()

print("Growing Season Summary (April - September 2025):")
print(summary.to_string(index=False))

## Key Findings

### Spatial Resolution Note

**Important:** NASA POWER data is available at **0.5° resolution** (approximately 55km grid cells). All 5 Nebraska fields fall within the same grid cell, so they share identical weather data.

This explains why all fields show:
- Identical temperature profiles
- Same total GDD (2253.73)
- Equal precipitation (620.49 mm)

### Recommendations for Field-Specific Weather

For more granular weather data, consider:
- **NOAA weather stations** - typically 1-10 km resolution
- **PRISM climate data** - 4 km resolution
- **On-site weather sensors** - point-specific data
- **Weather API services** (Weather Underground, OpenWeatherMap)

### Corn GDD Requirements (for reference)

| Growth Stage | GDD Required | Cumulative |
|--------------|--------------|------------|
| Emergence | 120 | 120 |
| V6 (6 leaves) | 500 | 620 |
| V12 (12 leaves) | 700 | 1320 |
| VT (tasseling) | 600 | 1920 |
| R1 (silking) | 200 | 2120 |
| Dent (R5) | 700 | 2820 |
| Maturity (R6) | 200 | 3020 |

The 2025 GDD of 2253.73 falls between VT (tasseling) and R1 (silking), indicating moderate heat accumulation for the year.

In [ ]:
# Save summary to CSV
summary.to_csv(DATA_DIR / 'weather_stats.csv', index=False)
print(f"Saved seasonal summary to: {DATA_DIR / 'weather_stats.csv'}")

## Output Files

### Data Files
- `data/weather_5fields_2025.csv` - Daily weather data (1825 rows)
- `data/weather_stats.csv` - Seasonal summaries

### Visualization Files
- `data/temperature_all_fields.png` - Daily temperature range
- `data/gdd_accumulation.png` - GDD accumulation curves
- `data/precipitation_cumulative.png` - Cumulative precipitation
- `data/precipitation_monthly.png` - Monthly precipitation bars